# `visualize_dashboard.ipynb` - Logica del cuadro de mando

Componente de visualizacion (independiente del resto, ver "Estructura y
Principios de Diseno" del enunciado): funciones de lectura de las tablas
Gold y de construccion de las figuras Plotly y KPIs que consume el cuadro
de mando interactivo de Streamlit.

> **Decision de diseno**: este notebook contiene y documenta, paso a paso,
> toda la logica de acceso a datos y construccion de figuras (consultas
> DuckDB sobre Parquet, figuras Plotly, KPIs). El proceso que realmente
> lanza `streamlit run` tiene que ser forzosamente un fichero `.py`
> (`dashboard/app.py`): Streamlit ejecuta su propio script de arriba a
> abajo en cada interaccion del usuario (cada clic, cada cambio de
> ciudad), y hacerlo invocando un kernel de Jupyter por cada rerun (via
> `nbclient`, como hace `cli.py`) seria demasiado lento para un panel
> interactivo; y la libreria `importnb` no puede importar este notebook
> tal cual porque su celda de encadenado usa `%run`, una magia de linea
> de IPython que no es Python valido fuera de un kernel. Por ambos
> motivos, `dashboard/app.py` reimplementa estas mismas consultas y
> figuras en un script minimo y autocontenido (usando `importnb` solo
> para reutilizar `config.ipynb`, que no tiene celdas `%run` y si es
> importable sin problemas). Los dos ficheros se mantienen sincronizados
> manualmente; cualquier cambio en una metrica debe reflejarse en ambos.

> Depende de: `config.ipynb`. Lee directamente de `data/gold/` (no
> depende de que `gold_analytics.ipynb` se acabe de ejecutar en el mismo
> kernel).

In [1]:
import os
if os.getcwd().endswith("climate_engine"):
    os.chdir("..")
%run climate_engine/config.ipynb

Fichero de configuracion: /root/project/Entregable_Proyecto_ClimateEngine_Ortega_Calvo_Francisco/config.yaml
Raiz del proyecto:        /root/project/Entregable_Proyecto_ClimateEngine_Ortega_Calvo_Francisco
Ubicaciones cargadas (5): ['Madrid', 'Barcelona', 'Valencia', 'Sevilla', 'Bilbao']
Variables horarias:       ['temperature_2m', 'relative_humidity_2m', 'precipitation', 'wind_speed_10m', 'surface_pressure']
Ventana de prediccion:    14 dias, zona horaria UTC
Politica de reintentos:   3 reintentos, backoff x2, codigos [429, 500, 502, 503, 504]
Rango temperatura valido: (-50.0, 60.0)
Rango humedad valido:     (0.0, 100.0)
Ruta bronze:  /root/project/Entregable_Proyecto_ClimateEngine_Ortega_Calvo_Francisco/data/bronze  (existe: True)
Ruta silver:  /root/project/Entregable_Proyecto_ClimateEngine_Ortega_Calvo_Francisco/data/silver  (existe: True)
Ruta gold:    /root/project/Entregable_Proyecto_ClimateEngine_Ortega_Calvo_Francisco/data/gold  (existe: True)
URL base API: https://api.open-me

In [2]:
import duckdb
import pandas as pd
import plotly.graph_objects as go

## Funcion principal

In [3]:
def _leer_tabla_gold(nombre_tabla):
    """Lee una tabla Gold completa (todas las ciudades) reconstruyendo la
    columna 'ciudad' a partir del particionado Hive de las carpetas
    (ciudad=<Nombre>), via DuckDB.

    Devuelve: DataFrame de pandas.
    """
    patron = str(RUTA_GOLD / nombre_tabla / "ciudad=*" / "datos.parquet")
    con = duckdb.connect(database=":memory:")
    return con.execute(f"SELECT * FROM read_parquet('{patron}', hive_partitioning=1)").fetchdf()


def cargar_resumen_diario():
    """Carga la tabla Gold de resumen diario por ciudad."""
    return _leer_tabla_gold("resumen_diario")


def cargar_media_movil():
    """Carga la tabla Gold de media movil de 24h de temperatura."""
    return _leer_tabla_gold("media_movil_temp")


def cargar_zscore():
    """Carga la tabla Gold de Z-Score horario de temperatura."""
    return _leer_tabla_gold("zscore_horario")


def ciudades_disponibles(df_resumen_diario):
    """Devuelve la lista ordenada de ciudades presentes en el resumen diario."""
    return sorted(df_resumen_diario["ciudad"].unique().tolist())


def construir_figura_temperatura(df_media_movil, ciudad):
    """Construye la figura Plotly de serie temporal: temperatura horaria
    real vs. su media movil de 24h, para una ciudad concreta.
    """
    datos_ciudad = df_media_movil[df_media_movil["ciudad"] == ciudad].sort_values("fecha_hora_utc")
    figura = go.Figure()
    figura.add_trace(go.Scatter(
        x=datos_ciudad["fecha_hora_utc"], y=datos_ciudad["temperature_2m"],
        mode="lines", name="Temperatura horaria (C)", line=dict(width=1),
    ))
    figura.add_trace(go.Scatter(
        x=datos_ciudad["fecha_hora_utc"], y=datos_ciudad["media_movil_24h_c"],
        mode="lines", name="Media movil 24h (C)", line=dict(width=3),
    ))
    figura.update_layout(
        title=f"Temperatura horaria y media movil 24h - {ciudad}",
        xaxis_title="Fecha y hora (UTC)", yaxis_title="Temperatura (C)",
        legend=dict(orientation="h", yanchor="bottom", y=1.02),
    )
    return figura


def construir_figura_zscore(df_zscore, ciudad):
    """Construye la figura Plotly del Z-Score horario de temperatura de
    una ciudad frente a su historico cargado, con lineas de referencia en
    +/-2 desviaciones tipicas.
    """
    datos_ciudad = df_zscore[df_zscore["ciudad"] == ciudad].sort_values("fecha_hora_utc")
    figura = go.Figure()
    figura.add_trace(go.Scatter(
        x=datos_ciudad["fecha_hora_utc"], y=datos_ciudad["z_score_temperatura"],
        mode="lines", name="Z-Score temperatura",
    ))
    figura.add_hline(y=2, line_dash="dash", line_color="red")
    figura.add_hline(y=-2, line_dash="dash", line_color="red")
    figura.update_layout(
        title=f"Z-Score horario de temperatura frente al historico - {ciudad}",
        xaxis_title="Fecha y hora (UTC)", yaxis_title="Z-Score",
    )
    return figura


def calcular_kpis(df_resumen_diario, ciudad):
    """Calcula los KPIs del ultimo dia disponible en el resumen diario
    para una ciudad: temperatura media/min/max, lluvia total y viento
    medio.

    Devuelve: diccionario de KPIs, o None si no hay datos para la ciudad.
    """
    datos_ciudad = df_resumen_diario[df_resumen_diario["ciudad"] == ciudad].sort_values("fecha")
    if datos_ciudad.empty:
        return None
    ultimo_dia = datos_ciudad.iloc[-1]
    return {
        "fecha": ultimo_dia["fecha"],
        "temperatura_media_c": ultimo_dia["temperatura_media_c"],
        "temperatura_minima_c": ultimo_dia["temperatura_minima_c"],
        "temperatura_maxima_c": ultimo_dia["temperatura_maxima_c"],
        "precipitacion_total_mm": ultimo_dia["precipitacion_total_mm"],
        "viento_medio_kmh": ultimo_dia["viento_medio_kmh"],
    }

## Prueba rapida

Se cargan las tres tablas Gold reales y se construyen, para Madrid, la figura de temperatura y la de Z-Score, ademas de los KPIs del ultimo dia, para confirmar que el dashboard tendra datos reales que mostrar.

In [4]:
df_resumen_diario_prueba = cargar_resumen_diario()
df_media_movil_prueba = cargar_media_movil()
df_zscore_prueba = cargar_zscore()

print(f"Ciudades disponibles: {ciudades_disponibles(df_resumen_diario_prueba)}")

kpis_madrid = calcular_kpis(df_resumen_diario_prueba, "Madrid")
print(f"\nKPIs del ultimo dia disponible en Madrid: {kpis_madrid}")

figura_temp = construir_figura_temperatura(df_media_movil_prueba, "Madrid")
figura_z = construir_figura_zscore(df_zscore_prueba, "Madrid")
print(f"\nFigura de temperatura construida con {len(figura_temp.data)} trazas.")
print(f"Figura de Z-Score construida con {len(figura_z.data)} trazas.")

assert kpis_madrid is not None, "Deberia haber KPIs disponibles para Madrid."
assert len(figura_temp.data) == 2, "La figura de temperatura debe tener 2 series (real + media movil)."
print("\nValidacion de la logica del dashboard: OK")

Ciudades disponibles: ['Barcelona', 'Bilbao', 'Madrid', 'Sevilla', 'Valencia']

KPIs del ultimo dia disponible en Madrid: {'fecha': Timestamp('2026-10-06 00:00:00'), 'temperatura_media_c': np.float64(23.000000000000004), 'temperatura_minima_c': np.float64(17.8), 'temperatura_maxima_c': np.float64(28.8), 'precipitacion_total_mm': np.float64(4.3999999999999995), 'viento_medio_kmh': np.float64(11.6)}

Figura de temperatura construida con 2 trazas.
Figura de Z-Score construida con 1 trazas.

Validacion de la logica del dashboard: OK


---
✅ **`visualize_dashboard.ipynb` verificado/a.**